In [1]:
import boto3
import json
from dotenv import load_dotenv
import os
from transformers import  AutoTokenizer
import pandas as pd
import random

from botocore.exceptions import ClientError

In [2]:
load_dotenv()

True

In [3]:
AWS_KEY = os.getenv("AWS_KEY")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_KEY")

In [4]:
bedrock = boto3.client(service_name='bedrock', 
region_name='us-west-2', 
aws_access_key_id=AWS_KEY, 
aws_secret_access_key=AWS_SECRET_KEY)

In [5]:
response = bedrock.list_foundation_models(byProvider="meta")

for summary in response["modelSummaries"]:
    print(summary["modelId"])

meta.llama2-13b-chat-v1:0:4k
meta.llama2-13b-chat-v1
meta.llama2-70b-chat-v1:0:4k
meta.llama2-70b-chat-v1
meta.llama2-13b-v1:0:4k
meta.llama2-13b-v1
meta.llama2-70b-v1:0:4k
meta.llama2-70b-v1
meta.llama3-8b-instruct-v1:0
meta.llama3-70b-instruct-v1:0
meta.llama3-1-8b-instruct-v1:0
meta.llama3-1-70b-instruct-v1:0
meta.llama3-1-405b-instruct-v1:0


# Test Model Response

In [6]:
model_id = 'meta.llama3-1-405b-instruct-v1:0'
hf_model_name = "meta-llama/Meta-Llama-3.1-405B-Instruct"

In [7]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(hf_model_name, padding_side="left")
tokenizer.pad_token = tokenizer.bos_token

In [8]:
temperature = 0.8
top_p=0.9
max_token_to_generate = 2000

In [9]:
def load_questions_to_df(question_file: str):
    """Load questions from a file into a DataFrame."""
    questions = []
    with open(question_file, "r") as ques_file:
        for line in ques_file:
            if line:
                questions.append(json.loads(line))
    
    df = pd.DataFrame([{
        "question_id": question["question_id"],
        "content": question["turns"][0]["content"],
        "cluster": question["cluster"]
    } for question in questions])
    
    return df

In [10]:
def prepare_prompts(df):
    """Prepare prompts dynamically based on question DataFrame."""
    prompts = []

    for _, row in df.iterrows():
        system_prompt = f"""
        You are a sophisticated AI-Expert there to help users solve tasks in several domains efficiently and accurately.
        Now solve the following task from the domain "{row['cluster']}".\n
        """

        user_message = f"{row['content']}"
        
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ]

        prompts.append(messages)

    df['prompt'] = prompts
    return df

In [11]:
# Load questions into a DataFrame
question_df = load_questions_to_df("arena-hard-auto/data/arena-hard-v0.1/question.jsonl")

# Prepare prompts
question_df = prepare_prompts(question_df)

# Select a random row from the DataFrame
example_question_df = question_df.sample(n=1, random_state=random.randint(0, len(question_df) - 1))

example_question_df

,question_id,content,cluster,prompt
298,77fd22f6615549cc8ddd2fdd56e80cd1,"if I have the numbers 1, 5, 6, 7, 9 and 10, wh...",Number Substitution Patterns,"[{'role': 'system', 'content': ' You a..."


In [12]:
message = example_question_df['prompt'].values[0]

prompt = tokenizer.apply_chat_template(
    message, 
    tokenize=False, 
    add_generation_prompt=True
)
print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a sophisticated AI-Expert there to help users solve tasks in several domains efficiently and accurately.
        Now solve the following task from the domain "Number Substitution Patterns".<|eot_id|><|start_header_id|>user<|end_header_id|>

if I have the numbers 1, 5, 6, 7, 9 and 10, what series of operations do I need to do to get 633 as result? The available operations are addition, substraction, multiplication and division. The use of all the numbers is not required but each number can only be used once.<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [13]:
body = json.dumps({
    "prompt": prompt,
    "max_gen_len":max_token_to_generate,
    "temperature":temperature,
    "top_p":top_p
})

In [14]:
bedrock_runtime = boto3.client(service_name='bedrock-runtime', 
                            region_name='us-west-2', 
                            aws_access_key_id=AWS_KEY, 
                            aws_secret_access_key=AWS_SECRET_KEY)

In [15]:
response = bedrock_runtime.invoke_model(body=body, modelId=model_id, accept="application/json", contentType="application/json")
print(response)

{'ResponseMetadata': {'RequestId': '04b883f5-2c51-416d-9c47-822ee92aef6e', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sun, 11 Aug 2024 16:41:19 GMT', 'content-type': 'application/json', 'content-length': '2816', 'connection': 'keep-alive', 'x-amzn-requestid': '04b883f5-2c51-416d-9c47-822ee92aef6e', 'x-amzn-bedrock-invocation-latency': '58883', 'x-amzn-bedrock-output-token-count': '917', 'x-amzn-bedrock-input-token-count': '140'}, 'RetryAttempts': 2}, 'contentType': 'application/json', 'body': <botocore.response.StreamingBody object at 0x17af55a50>}


In [16]:
# Colors to test the model
class bcolors:
    OKGREEN = '\033[92m'
    CBLUE   = '\33[34m'
    CVIOLET = '\33[35m'
    ENDC = '\033[0m'

In [17]:
def print_prompt(prompt, response, with_system=False):
    print("="*30 + f" Chat with  --- {model_id} ---  LLM using AWS Model API " + "="*30 + "\n")
    for idx, message in enumerate(prompt):
        if prompt[idx]['role'] == 'user':
            color = bcolors.CBLUE
            print(color + f"[ {prompt[idx]['role'].upper()} ]" + bcolors.ENDC)
            print(prompt[idx]['content']+ "\n")
        else: 
            if with_system:
                color = bcolors.CVIOLET
                print(color + f"[ {prompt[idx]['role'].upper()} ]" + bcolors.ENDC)
                print(prompt[idx]['content']+ "\n")

    print(bcolors.OKGREEN + f"[ {model_id} ]" + bcolors.ENDC)
    print(response["generation"])

In [18]:
model_response = json.loads(response["body"].read())

In [19]:
print_prompt(message, model_response, with_system=True)

============================== Chat with  --- meta.llama3-1-405b-instruct-v1:0 ---  LLM using AWS Model API ==============================

[ SYSTEM ]

        You are a sophisticated AI-Expert there to help users solve tasks in several domains efficiently and accurately.
        Now solve the following task from the domain "Number Substitution Patterns".

        

[ USER ]
if I have the numbers 1, 5, 6, 7, 9 and 10, what series of operations do I need to do to get 633 as result? The available operations are addition, substraction, multiplication and division. The use of all the numbers is not required but each number can only be used once.

[ meta.llama3-1-405b-instruct-v1:0 ]
To find a series of operations that yields 633 using the numbers 1, 5, 6, 7, 9, and 10, we can try different combinations. Here's one possible solution:

1. Multiply 10 and 63 (but we don't have 63) - instead, create 63 by:
   - Multiply 7 and 9 to get 63.

However, we need 633, not 63. Since we have a 10, let'

# Test Response Stream

In [21]:
def print_stream(prompt, streaming_response, with_system=False):
    print("="*30 + f" Chat with  --- {model_id} ---  LLM using vLLM " + "="*30 + "\n")
    for idx, message in enumerate(prompt):
        if prompt[idx]['role'] == 'user':
            color = bcolors.CBLUE
            print(color + f"[ {prompt[idx]['role'].upper()} ]" + bcolors.ENDC)
            print(prompt[idx]['content']+ "\n")
        else: 
            if with_system:
                color = bcolors.CVIOLET
                print(color + f"[ {prompt[idx]['role'].upper()} ]" + bcolors.ENDC)
                print(prompt[idx]['content']+ "\n")

    print(bcolors.OKGREEN + f"[ ASSISTANT ]" + bcolors.ENDC + "\n")
    for event in streaming_response["body"]:
        chunk = json.loads(event["chunk"]["bytes"])
        if "generation" in chunk:
            print(chunk["generation"], end="")

In [22]:
# Select a random row from the DataFrame
example_question_df = question_df.sample(n=1, random_state=random.randint(0, len(question_df) - 1))

message = example_question_df['prompt'].values[0]

prompt = tokenizer.apply_chat_template(
    message, 
    tokenize=False, 
    add_generation_prompt=True
)
print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a sophisticated AI-Expert there to help users solve tasks in several domains efficiently and accurately.
        Now solve the following task from the domain "English Longest Words Inquiry".<|eot_id|><|start_header_id|>user<|end_header_id|>

Please write a Python function that receives a data frame with columns date and winner and returns the longest number of consecutive win by Alice<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [23]:
# Format the request payload using the model's native structure.
native_request = {
    "prompt": prompt,
    "max_gen_len": 2048,
    "temperature": 0.8,
    "top_p": 0.8
}

# Convert the native request to JSON.
request = json.dumps(native_request)

try:
    # Invoke the model with the request.
    streaming_response = bedrock_runtime.invoke_model_with_response_stream(
        modelId=model_id, body=request
    )
    print_stream(message, streaming_response, with_system=True)

except (ClientError, Exception) as e:
    print(f"ERROR: Can't invoke '{model_id}'. Reason: {e}")
    exit(1)

============================== Chat with  --- meta.llama3-1-405b-instruct-v1:0 ---  LLM using vLLM ==============================

[ SYSTEM ]

        You are a sophisticated AI-Expert there to help users solve tasks in several domains efficiently and accurately.
        Now solve the following task from the domain "English Longest Words Inquiry".

        

[ USER ]
Please write a Python function that receives a data frame with columns date and winner and returns the longest number of consecutive win by Alice

[ ASSISTANT ]

Here is a Python function that solves the problem. It assumes that the input DataFrame has a 'winner' column and a 'date' column, and that the 'date' column is in a format that can be sorted.

```python
import pandas as pd

def longest_consecutive_wins(df):
    """
    This function calculates the longest number of consecutive wins by 'Alice'.

    Parameters:
    df (pandas DataFrame): A DataFrame with 'date' and 'winner' columns.

    Returns:
    int: The longe

# Test Mistral

In [28]:
model_id = 'mistral.mistral-large-2407-v1:0'

In [36]:
def print_mistral_prompt(prompt, response, with_system=False):
    print("="*30 + f" Chat with  --- {model_id} ---  LLM using AWS Model API " + "="*30 + "\n")
    for idx, message in enumerate(prompt):
        if prompt[idx]['role'] == 'user':
            color = bcolors.CBLUE
            print(color + f"[ {prompt[idx]['role'].upper()} ]" + bcolors.ENDC)
            print(prompt[idx]['content']+ "\n")
        else: 
            if with_system:
                color = bcolors.CVIOLET
                print(color + f"[ {prompt[idx]['role'].upper()} ]" + bcolors.ENDC)
                print(prompt[idx]['content']+ "\n")

    print(bcolors.OKGREEN + f"[ {model_response['choices'][0]['message']['role']} ]" + bcolors.ENDC)
    print(model_response['choices'][0]['message']['content'])

In [24]:
# Select a random row from the DataFrame
example_question_df = question_df.sample(n=1, random_state=random.randint(0, len(question_df) - 1))

message = example_question_df['prompt'].values[0]

print(message)

[{'role': 'system', 'content': '\n        You are a sophisticated AI-Expert there to help users solve tasks in several domains efficiently and accurately.\n        Now solve the following task from the domain "Cube, Shaking, Box Dynamics".\n\n        '}, {'role': 'user', 'content': "youll be acting as a senior analyst who is an expert in sql. youll be helping me, a junior analyst understand sql queries well use together. can you add comments to this query to make it easy for other analysts to understand? SELECT ifnull(region,'') region,ifnull(subregion,'') subregion,\navg(COUNT_SERVICE_LINES_USED) avg_ct_sl,count(DISTINCT patientid) ct_patients \nFROM PATIENT_INFO\nGROUP BY cube(1,2) ORDER BY avg_ct_sl DESC"}]


In [29]:
body = json.dumps({
            "max_tokens": max_token_to_generate,
            "temperature":temperature,
            "messages": message
        })

In [30]:
response = bedrock_runtime.invoke_model(body=body, modelId=model_id, accept="application/json", contentType="application/json")
print(response)

{'ResponseMetadata': {'RequestId': '0ed32011-e930-48b6-be8a-126035b08f80', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sat, 10 Aug 2024 21:17:42 GMT', 'content-type': 'application/json', 'content-length': '2247', 'connection': 'keep-alive', 'x-amzn-requestid': '0ed32011-e930-48b6-be8a-126035b08f80', 'x-amzn-bedrock-invocation-latency': '16644', 'x-amzn-bedrock-output-token-count': '626', 'x-amzn-bedrock-input-token-count': '184'}, 'RetryAttempts': 0}, 'contentType': 'application/json', 'body': <botocore.response.StreamingBody object at 0x1554d3df0>}


In [31]:
model_response = json.loads(response["body"].read())


{'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "Certainly! Here is the SQL query with added comments to help you understand each part of the query:\n\n```sql\n-- Selecting the specified columns with fallback values for nulls and calculating averages\nSELECT\n    -- If region is null, return an empty string instead\n    IFNULL(region, '') AS region,\n    -- If subregion is null, return an empty string instead\n    IFNULL(subregion, '') AS subregion,\n    -- Calculate the average of COUNT_SERVICE_LINES_USED\n    AVG(COUNT_SERVICE_LINES_USED) AS avg_ct_sl,\n    -- Count the distinct number of patient IDs\n    COUNT(DISTINCT patientid) AS ct_patients\nFROM\n    -- Specifying the table to select data from\n    PATIENT_INFO\n-- Grouping the results by multiple combinations of region and subregion\nGROUP BY\n    -- The CUBE function allows for multiple groupings, including region, subregion, and both\n    CUBE(1, 2)\n-- Ordering the results in descending order by the av

In [37]:
print_mistral_prompt(message, model_response, with_system=True)

============================== Chat with  --- mistral.mistral-large-2407-v1:0 ---  LLM using AWS Model API ==============================

[ SYSTEM ]

        You are a sophisticated AI-Expert there to help users solve tasks in several domains efficiently and accurately.
        Now solve the following task from the domain "Cube, Shaking, Box Dynamics".

        

[ USER ]
youll be acting as a senior analyst who is an expert in sql. youll be helping me, a junior analyst understand sql queries well use together. can you add comments to this query to make it easy for other analysts to understand? SELECT ifnull(region,'') region,ifnull(subregion,'') subregion,
avg(COUNT_SERVICE_LINES_USED) avg_ct_sl,count(DISTINCT patientid) ct_patients 
FROM PATIENT_INFO
GROUP BY cube(1,2) ORDER BY avg_ct_sl DESC

[ assistant ]
Certainly! Here is the SQL query with added comments to help you understand each part of the query:

```sql
-- Selecting the specified columns with fallback values for nulls and c